# Agreement of LLM Judges in Persona Contradiction Detection

**Author:** Matyáš Martinek
**Course:** Probability and Statistics
**Language:** Python

## Objective

This project investigates whether different large language models agree when
judging contradictions between a complete persona and a dialogue.

Each LLM judge receives the same persona–dialogue example and predicts one of
two labels:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary goal is to measure inter-judge agreement and determine whether
different LLM judges exhibit systematically different decision behaviour.

The project further investigates whether agreement is related to properties
of the input, including persona length, dialogue length, lexical overlap, and
the number of augmented persona facts.

## 1. Data source and provenance

The base data come from the **PersonaChat** dataset introduced by
Zhang et al. (2018). PersonaChat contains multi-turn English dialogues in
which each participant is assigned a persona represented by several natural
language statements.

For this project, I use the PersonaChat distribution available on Hugging Face:

https://huggingface.co/datasets/awsaf49/persona-chat

The original dataset is described in:

> Zhang, S., Dinan, E., Urbanek, J., Szlam, A., Kiela, D., & Weston, J.
> (2018). *Personalizing Dialogue Agents: I have a dog, do you have pets too?*
> Proceedings of ACL 2018, pp. 2204–2213.
> DOI: 10.18653/v1/P18-1205.

Original paper:

https://aclanthology.org/P18-1205/

### Derived data

The PersonaChat data were processed as part of a bachelor-thesis project on
persona consistency.

Persona facts were automatically grounded in their corresponding dialogues.
Grounding estimates whether a persona fact is supported by evidence in the
dialogue.

Contradiction-oriented persona variants were subsequently generated using
deterministic rule-based augmentations. Multiple supported facts belonging to
the same persona may be modified simultaneously while the corresponding
dialogue remains unchanged.

The resulting profile-level dataset is stored in:

`personachat_augmented_v2.parquet`

Each row represents one speaker persona and its dialogue and contains:

- the complete original persona,
- the complete augmented persona,
- the unchanged dialogue,
- the number and types of applied augmentations,
- grounding information for the changed facts,
- augmentation provenance.

For rule-based profiles, an augmented persona is assigned the expected relation
`contradiction` when at least one changed persona fact is grounded in the
dialogue.

These automatically derived relations are treated as **expected labels**, not
as manually verified ground truth.

In [19]:
from pathlib import Path
import re

import pandas as pd

from judge_config import (
    PROMPT_ID,
    build_judge_prompt,
    parse_judge_output,
)


RANDOM_SEED = 42

DATA_DIR = Path("data")
JUDGMENTS_DIR = DATA_DIR / "outputs"

SOURCE_DATA_PATH = (
    DATA_DIR
    / "personachat_augmented_v2.parquet"
)

source_df = pd.read_parquet(
    SOURCE_DATA_PATH
)

print(f"Rows: {len(source_df):,}")
print(
    f"Dialogues: "
    f"{source_df['dialogue_id'].nunique():,}"
)

print(
    "\nAugmentation families:"
)

print(
    source_df[
        "augmentation_family"
    ].value_counts()
)

source_df.head()

Rows: 26,368
Dialogues: 8,939

Augmentation families:
augmentation_family
persona_swap    17878
rule             8490
Name: count, dtype: int64


,augmentation_id,dialogue_id,speaker,augmentation_family,augmentation_type,augmentation_method,dialogue_text,original_persona,augmented_persona,persona_fact_count,...,contradiction_strengths,generation_notes,rule_versions,changed_expected_relations,changed_grounding_labels,changed_grounding_scores,changed_best_utterances,expected_relation,donor_dialogue_id,seed
0,aug_000000,0,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing ? i am getting re...","[i like canning and whittling., to stay in sha...","[i do not like canning and whittling., to stay...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9796050786972046],[i am ! for my hobby i like to do canning or s...,contradiction,NaN,42
1,aug_000001,0,speaker_2,rule,preference_negation,rule_based,"PARTNER: hi , how are you doing ? i am getting...","[i like to remodel homes., i like to go huntin...","[i do not like to remodel homes., i do not lik...",4,...,"[strong, strong]",[Preference polarity was negated using a deter...,"[v1, v1]","[contradiction, contradiction]","[grounded, grounded]","[0.8077989220619202, 0.8201733231544495]",[i also remodel homes when i am not out bow hu...,contradiction,NaN,42
2,aug_000002,1,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing today ?\nPARTNER:...","[i wish i could live forever., i only date peo...","[i wish i could live forever., i only date peo...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9774878025054932],"[i really enjoy free diving , how about you , ...",contradiction,NaN,42
3,aug_000003,1,speaker_2,rule,family_composition_swap,rule_based,"PARTNER: hi , how are you doing today ?\nSELF:...","[my mom is my best friend., i have four sister...","[my mom is my best friend., i have 2 sisters.,...",4,...,[medium_strong],[Family composition count was replaced with a ...,[v1],[contradiction],[grounded],[0.9484180808067322],[i am spending time with my 4 sisters what are...,contradiction,NaN,42
4,aug_000004,2,speaker_1,rule,preference_negation,rule_based,"SELF: we all live in a yellow submarine , a ye...","[i love the beatles., i have trouble getting a...","[i hate the beatles., i have trouble getting a...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.8553723692893982],"[lol . i am shy , anything to break the ice , ...",contradiction,NaN,42


In [20]:
eligible_df = source_df[
    source_df["augmentation_family"].eq("rule")
    & source_df["expected_relation"].eq("contradiction")
    & source_df["num_grounded_augmented_facts"].gt(0)
].copy()


print(
    f"Eligible rule-augmented profiles: "
    f"{len(eligible_df):,}"
)

print(
    f"Unique dialogues: "
    f"{eligible_df['dialogue_id'].nunique():,}"
)

Eligible rule-augmented profiles: 8,469
Unique dialogues: 6,399


In [21]:
profile_summary = pd.DataFrame(
    {
        "num_augmented_facts": (
            eligible_df[
                "num_augmented_facts"
            ].describe()
        ),
        "num_grounded_augmented_facts": (
            eligible_df[
                "num_grounded_augmented_facts"
            ].describe()
        ),
    }
)

profile_summary

,num_augmented_facts,num_grounded_augmented_facts
count,8469.000000,8469.000000
mean,1.502775,1.499351
std,0.740965,0.739694
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,5.000000,5.000000


## 2. Experimental dataset construction

The experiment is based on complete persona–dialogue profiles rather than
individual persona facts.

For every selected profile, two matched evaluation examples are constructed:

1. **Original variant** – the complete original persona is paired with its
   dialogue and is expected to be `NO_CONTRADICTION`.
2. **Augmented variant** – the complete augmented persona is paired with the
   same dialogue and is expected to be `CONTRADICTION`.

The dialogue is therefore identical within each pair. The only difference is
that one or more persona statements have been modified in the augmented
variant.

Only rule-based profiles with expected relation `contradiction` are used. Such
profiles contain at least one changed persona fact that was grounded in the
dialogue.

To reduce dependence between observations, at most one speaker profile is
selected from each dialogue. Sampling is deterministic using a fixed random
seed.

Persona-swap examples are excluded from the main experiment because they are
currently contradiction candidates rather than verified contradiction-oriented
profiles.

In [22]:
N_PAIRS = 1000

In [23]:
sampling_pool = (
    eligible_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .drop_duplicates(
        subset="dialogue_id"
    )
    .reset_index(drop=True)
)

if len(sampling_pool) < N_PAIRS:
    raise ValueError(
        f"Only {len(sampling_pool):,} unique dialogues "
        f"are available, but {N_PAIRS:,} pairs were requested."
    )

selected_pairs_df = (
    sampling_pool
    .head(N_PAIRS)
    .copy()
)

selected_pairs_df = selected_pairs_df.rename(
    columns={
        "num_augmented_facts": (
            "pair_num_augmented_facts"
        ),
        "num_grounded_augmented_facts": (
            "pair_num_grounded_augmented_facts"
        ),
    }
)

print(
    f"Selected profiles: "
    f"{len(selected_pairs_df):,}"
)

print(
    f"Unique dialogues: "
    f"{selected_pairs_df['dialogue_id'].nunique():,}"
)

Selected profiles: 1,000
Unique dialogues: 1,000


In [24]:
def format_persona(
    persona,
) -> str:
    """
    Format a list of persona statements for LLM evaluation.
    """
    return "\n".join(
        f"- {fact}"
        for fact in persona
    )

In [25]:
original_df = selected_pairs_df.copy()

original_df["variant"] = "original"
original_df["persona_text"] = (
    original_df[
        "original_persona"
    ].map(format_persona)
)

original_df["expected_label"] = (
    "NO_CONTRADICTION"
)

original_df["num_changed_facts"] = 0
original_df["num_grounded_changed_facts"] = 0

original_df["expected_label_binary"] = 0


augmented_df = selected_pairs_df.copy()

augmented_df["variant"] = "augmented"
augmented_df["persona_text"] = (
    augmented_df[
        "augmented_persona"
    ].map(format_persona)
)

augmented_df["expected_label"] = (
    "CONTRADICTION"
)

augmented_df["num_changed_facts"] = (
    augmented_df[
        "pair_num_augmented_facts"
    ]
)

augmented_df[
    "num_grounded_changed_facts"
] = (
    augmented_df[
        "pair_num_grounded_augmented_facts"
    ]
)

augmented_df["expected_label_binary"] = 1

In [26]:
pair_ids = [
    f"pair_{i:04d}"
    for i in range(
        len(selected_pairs_df)
    )
]

original_df["pair_id"] = pair_ids
augmented_df["pair_id"] = pair_ids

In [27]:
experiment_df = pd.concat(
    [
        original_df,
        augmented_df,
    ],
    ignore_index=True,
)

experiment_df["example_id"] = (
    experiment_df["pair_id"]
    + "_"
    + experiment_df["variant"]
)

experiment_df = experiment_df.rename(
    columns={
        "num_augmented_facts": (
            "pair_num_augmented_facts"
        ),
        "num_grounded_augmented_facts": (
            "pair_num_grounded_augmented_facts"
        ),
    }
)

experiment_df["num_changed_facts"] = (
    experiment_df["variant"]
    .eq("augmented")
    .astype(int)
    * experiment_df[
        "pair_num_augmented_facts"
    ]
)

experiment_df[
    "num_grounded_changed_facts"
] = (
    experiment_df["variant"]
    .eq("augmented")
    .astype(int)
    * experiment_df[
        "pair_num_grounded_augmented_facts"
    ]
)

In [28]:
experiment_df = (
    experiment_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

In [29]:
assert (
    len(experiment_df)
    == 2 * len(selected_pairs_df)
)

assert experiment_df[
    "example_id"
].is_unique

assert (
    experiment_df
    .groupby("pair_id")
    .size()
    .eq(2)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_text"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_id"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df[
        "pair_id"
    ].nunique()
    == experiment_df[
        "dialogue_id"
    ].nunique()
)

print(
    "Experimental dataset validation passed."
)

Experimental dataset validation passed.


### 2.1 Input characteristics

Several simple textual characteristics are computed before LLM evaluation.
These variables are later used to investigate whether judge decisions or
inter-judge disagreement depend on properties of the input.

The following characteristics are considered:

- complete persona length,
- number of persona facts,
- dialogue length,
- total textual input length,
- lexical overlap between the complete persona and the dialogue.

For augmented profiles, the source data additionally provide the number of
changed persona facts and the number of changed facts grounded in the dialogue.

Lengths are measured in word tokens using a simple deterministic tokenizer.
Lexical overlap is measured using Jaccard similarity between the sets of words
appearing in the persona and dialogue.

In [30]:
WORD_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize_words(text: str) -> list[str]:
    """Return lowercase word tokens from English text."""
    return WORD_PATTERN.findall(str(text).lower())


def jaccard_overlap(text_a: str, text_b: str) -> float:
    """Compute Jaccard similarity between unique word sets."""
    words_a = set(tokenize_words(text_a))
    words_b = set(tokenize_words(text_b))

    union = words_a | words_b

    if not union:
        return 0.0

    return len(words_a & words_b) / len(union)

In [31]:
experiment_df[
    "persona_length"
] = (
    experiment_df[
        "persona_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "dialogue_length"
] = (
    experiment_df[
        "dialogue_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "input_length"
] = (
    experiment_df[
        "persona_length"
    ]
    + experiment_df[
        "dialogue_length"
    ]
)

experiment_df[
    "lexical_overlap"
] = experiment_df.apply(
    lambda row: jaccard_overlap(
        row["persona_text"],
        row["dialogue_text"],
    ),
    axis=1,
)

experiment_df[
    [
        "persona_fact_count",
        "persona_length",
        "dialogue_length",
        "input_length",
        "lexical_overlap",
        "pair_num_augmented_facts",
        "pair_num_grounded_augmented_facts",
        "num_changed_facts",
        "num_grounded_changed_facts",
    ]
].describe()

,persona_fact_count,persona_length,dialogue_length,input_length,lexical_overlap,pair_num_augmented_facts,pair_num_grounded_augmented_facts,num_changed_facts,num_grounded_changed_facts
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,4.540000,26.821500,165.115000,191.936500,0.141222,1.494000,1.491000,0.747000,0.745500
std,0.508458,6.706648,28.514247,29.638061,0.043630,0.687168,0.685678,0.891285,0.889452
min,3.000000,11.000000,74.000000,99.000000,0.025424,1.000000,1.000000,0.000000,0.000000
25%,4.000000,22.000000,146.000000,172.000000,0.111111,1.000000,1.000000,0.000000,0.000000
50%,5.000000,26.000000,164.000000,192.000000,0.135593,1.000000,1.000000,0.500000,0.500000
75%,5.000000,31.000000,184.000000,210.000000,0.165158,2.000000,2.000000,1.000000,1.000000
max,5.000000,67.000000,329.000000,352.000000,0.328947,4.000000,4.000000,4.000000,4.000000


In [32]:
experiment_summary = pd.Series(
    {
        "examples": (
            len(experiment_df)
        ),
        "pairs": (
            experiment_df[
                "pair_id"
            ].nunique()
        ),
        "dialogues": (
            experiment_df[
                "dialogue_id"
            ].nunique()
        ),
        "original_examples": (
            experiment_df[
                "variant"
            ].eq("original").sum()
        ),
        "augmented_examples": (
            experiment_df[
                "variant"
            ].eq("augmented").sum()
        ),
    },
    name="count",
)

experiment_summary.to_frame()

,count
examples,2000
pairs,1000
dialogues,1000
original_examples,1000
augmented_examples,1000


In [33]:
pd.crosstab(
    selected_pairs_df[
        "pair_num_augmented_facts"
    ],
    selected_pairs_df[
        "pair_num_grounded_augmented_facts"
    ],
    rownames=["augmented facts"],
    colnames=["grounded augmented facts"],
)

grounded augmented facts,1.0,2.0,3.0,4.0
augmented facts,,,,
1.0,606,0,0,0
2.0,2,303,0,0
3.0,0,1,77,0
4.0,0,0,0,11


In [34]:
pd.crosstab(
    selected_pairs_df["pair_num_augmented_facts"],
    selected_pairs_df["pair_num_grounded_augmented_facts"],
)

pair_num_grounded_augmented_facts,1.0,2.0,3.0,4.0
pair_num_augmented_facts,,,,
1.0,606,0,0,0
2.0,2,303,0,0
3.0,0,1,77,0
4.0,0,0,0,11


### 2.2 Final experimental sample

The resulting experimental sample is treated as fixed for all subsequent LLM
evaluation.

The sample contains matched original and augmented persona–dialogue examples.
Both members of a pair contain exactly the same dialogue. The original variant
contains the complete original persona, while the augmented variant contains
the corresponding contradiction-oriented persona produced by the rule-based
augmentation pipeline.

The dataset is generated deterministically using a fixed random seed. No
examples are added, removed, or resampled based on the outputs of the LLM
judges.

In [35]:
EXPERIMENT_DATA_PATH = (
    DATA_DIR
    / "llm_judge_experiment_v2.parquet"
)

experiment_df.to_parquet(
    EXPERIMENT_DATA_PATH,
    index=False,
)

print(
    f"Saved {len(experiment_df):,} examples to "
    f"{EXPERIMENT_DATA_PATH}"
)

Saved 2,000 examples to data/llm_judge_experiment_v2.parquet


### 2.3 Frozen pilot sample


In [36]:
PILOT_PAIRS = 50

pilot_pair_ids = (
    experiment_df[
        ["pair_id"]
    ]
    .drop_duplicates()
    .sample(
        n=PILOT_PAIRS,
        random_state=RANDOM_SEED,
    )
    ["pair_id"]
)

pilot_df = (
    experiment_df[
        experiment_df[
            "pair_id"
        ].isin(pilot_pair_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(pilot_df) == 2 * PILOT_PAIRS

print(
    f"Pilot: {PILOT_PAIRS} pairs / "
    f"{len(pilot_df)} examples"
)

Pilot: 50 pairs / 100 examples



Before the full LLM evaluation, a fixed pilot sample of 50 matched pairs
is used to verify the prompt, inference pipeline, output parsing, and judge
behaviour.

The same frozen pilot examples are evaluated by all four judges.

In [37]:
PILOT_DATA_PATH = (
    DATA_DIR
    / "llm_judge_pilot_v1.parquet"
)

pilot_df.to_parquet(
    PILOT_DATA_PATH,
    index=False,
)

print(
    f"Saved {len(pilot_df)} pilot examples "
    f"({pilot_df['pair_id'].nunique()} pairs) "
    f"to {PILOT_DATA_PATH}"
)

Saved 100 pilot examples (50 pairs) to data/llm_judge_pilot_v1.parquet


## 3. LLM judges

The same experimental examples are evaluated independently by several
instruction-tuned large language models.

Each judge receives:

- the complete persona of the speaker labelled `SELF`,
- the corresponding dialogue,
- the same contradiction-detection instructions.

The model must return a binary decision:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary analysis compares decisions between judges rather than treating
the automatically generated expected labels as manually verified ground truth.

### 3.1 Evaluation prompt

All judges use the same prompt and binary decision rule.

A contradiction is present when something stated or clearly implied by `SELF`
is incompatible with at least one statement in the complete persona.

The prompt explicitly distinguishes:

- `SELF` from `PARTNER`,
- contradiction from missing evidence,
- incompatible information from additional compatible information,
- temporal and modal differences from genuine contradictions.

Judges are instructed to use only information contained in the persona and
dialogue and to return one of two labels:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The exact prompt and output parser are defined in `judge_config.py`.
The final prompt used in the experiment is identified by `full_persona_v3`.

In [38]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("original")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are a binary judge of persona consistency.

Determine whether statements made by SELF in the dialogue contradict at least one statement in SELF's persona.

Rules:
- CONTRADICTION: SELF explicitly states or clearly implies information that cannot be true at the same time as at least one persona statement.
- NO_CONTRADICTION: no such conflict is present.
- One conflicting persona statement is sufficient for CONTRADICTION.
- A persona statement that is not discussed in the dialogue is not a contradiction.
- Claims made only by PARTNER are not evidence about SELF. Use PARTNER's utterances only as conversational context.
- Respect temporal and modal information. Past and present statements, intentions, wishes, and current facts are contradictions only when they are genuinely incompatible.
- Additional compatible information is not a contradiction.
- Use only the information in the persona and dialogue. Do not assume unstated facts.

Persona of SELF:
- i need a wheel chair to get around.

In [39]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("augmented")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are a binary judge of persona consistency.

Determine whether statements made by SELF in the dialogue contradict at least one statement in SELF's persona.

Rules:
- CONTRADICTION: SELF explicitly states or clearly implies information that cannot be true at the same time as at least one persona statement.
- NO_CONTRADICTION: no such conflict is present.
- One conflicting persona statement is sufficient for CONTRADICTION.
- A persona statement that is not discussed in the dialogue is not a contradiction.
- Claims made only by PARTNER are not evidence about SELF. Use PARTNER's utterances only as conversational context.
- Respect temporal and modal information. Past and present statements, intentions, wishes, and current facts are contradictions only when they are genuinely incompatible.
- Additional compatible information is not a contradiction.
- Use only the information in the persona and dialogue. Do not assume unstated facts.

Persona of SELF:
- i do not like fantasizing.
- i like

### 3.2 Synthetic sanity check

Before interpreting disagreement on the PersonaChat pilot sample, the
inference pipeline is evaluated on a small manually constructed sanity set.

The sanity examples are deliberately simple and unambiguous. They are not
used as part of the main statistical analysis. Their purpose is to verify
that:

- the prompt is interpreted correctly,
- SELF and PARTNER are distinguished correctly,
- obvious contradictions can be detected,
- obvious consistent examples are not systematically labelled as
  contradictions,
- the inference and output-parsing pipeline behaves as expected.

Unlike the automatically derived PersonaChat reference labels, the labels in
this synthetic set are manually defined by construction.

In [80]:
SYNTHETIC_SANITY_DATA_PATH = (
    DATA_DIR
    / "llm_judge_synthetic_sanity_v1.parquet"
)


synthetic_examples = [
    # ---------------------------------------------------------
    # CONTRADICTION
    # ---------------------------------------------------------
    {
        "example_id": "synthetic_c01_preference",
        "sanity_category": "preference",
        "persona_text": (
            "- i hate chocolate.\n"
            "- i have a dog.\n"
            "- i live in boston."
        ),
        "dialogue_text": (
            "PARTNER: do you like chocolate?\n"
            "SELF: yes, i love chocolate and eat it every day."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "SELF explicitly says they love chocolate while "
            "the persona says they hate it."
        ),
    },
    {
        "example_id": "synthetic_c02_job",
        "sanity_category": "profession",
        "persona_text": (
            "- i work as a teacher.\n"
            "- i have a sister.\n"
            "- i enjoy hiking."
        ),
        "dialogue_text": (
            "PARTNER: are you still a teacher?\n"
            "SELF: no, i do not work as a teacher. "
            "i work as a nurse."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "SELF explicitly denies currently working as a teacher."
        ),
    },
    {
        "example_id": "synthetic_c03_location",
        "sanity_category": "location",
        "persona_text": (
            "- i live in paris.\n"
            "- i like jazz.\n"
            "- i own a bicycle."
        ),
        "dialogue_text": (
            "PARTNER: do you live in paris?\n"
            "SELF: no, i do not live in paris. i live in berlin."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "SELF explicitly denies living in the persona location."
        ),
    },
    {
        "example_id": "synthetic_c04_age",
        "sanity_category": "age",
        "persona_text": (
            "- i am 25 years old.\n"
            "- i enjoy cooking.\n"
            "- i have a cat."
        ),
        "dialogue_text": (
            "PARTNER: how old are you?\n"
            "SELF: i am 40 years old."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "The current ages are mutually incompatible."
        ),
    },
    {
        "example_id": "synthetic_c05_relationship",
        "sanity_category": "relationship_status",
        "persona_text": (
            "- i am single.\n"
            "- i play tennis.\n"
            "- i like coffee."
        ),
        "dialogue_text": (
            "PARTNER: are you single?\n"
            "SELF: no, i am married."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "SELF explicitly says they are married while "
            "the persona says they are single."
        ),
    },
    {
        "example_id": "synthetic_c06_pets",
        "sanity_category": "pets",
        "persona_text": (
            "- i have no pets.\n"
            "- i work from home.\n"
            "- i like movies."
        ),
        "dialogue_text": (
            "PARTNER: do you have any pets?\n"
            "SELF: yes, i have two dogs."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "Having two dogs contradicts having no pets."
        ),
    },
    {
        "example_id": "synthetic_c07_children",
        "sanity_category": "family",
        "persona_text": (
            "- i have no children.\n"
            "- i like gardening.\n"
            "- i live in a small town."
        ),
        "dialogue_text": (
            "PARTNER: do you have children?\n"
            "SELF: yes, i have three children."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "Having three children contradicts having no children."
        ),
    },
    {
        "example_id": "synthetic_c08_diet",
        "sanity_category": "diet",
        "persona_text": (
            "- i never eat meat.\n"
            "- i enjoy reading.\n"
            "- i have brown hair."
        ),
        "dialogue_text": (
            "PARTNER: what do you usually eat?\n"
            "SELF: i eat meat every day."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "Eating meat every day contradicts never eating meat."
        ),
    },
    {
        "example_id": "synthetic_c09_favorite",
        "sanity_category": "favorite",
        "persona_text": (
            "- my favorite color is blue.\n"
            "- i have one brother.\n"
            "- i enjoy swimming."
        ),
        "dialogue_text": (
            "PARTNER: what is your favorite color?\n"
            "SELF: my favorite color is red, not blue."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "SELF explicitly gives a different favorite color "
            "and rejects the persona value."
        ),
    },
    {
        "example_id": "synthetic_c10_activity",
        "sanity_category": "preference",
        "persona_text": (
            "- i do not like running.\n"
            "- i own a camera.\n"
            "- i drink tea."
        ),
        "dialogue_text": (
            "PARTNER: do you run?\n"
            "SELF: yes, i love running and i run every morning."
        ),
        "expected_label": "CONTRADICTION",
        "expected_label_binary": 1,
        "rationale": (
            "Loving running contradicts explicitly not liking running."
        ),
    },

    # ---------------------------------------------------------
    # NO_CONTRADICTION
    # ---------------------------------------------------------
    {
        "example_id": "synthetic_n01_preference",
        "sanity_category": "preference",
        "persona_text": (
            "- i hate chocolate.\n"
            "- i have a dog.\n"
            "- i live in boston."
        ),
        "dialogue_text": (
            "PARTNER: do you like chocolate?\n"
            "SELF: no, i hate chocolate."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "SELF directly confirms the persona preference."
        ),
    },
    {
        "example_id": "synthetic_n02_job",
        "sanity_category": "profession",
        "persona_text": (
            "- i work as a teacher.\n"
            "- i have a sister.\n"
            "- i enjoy hiking."
        ),
        "dialogue_text": (
            "PARTNER: what do you do for work?\n"
            "SELF: i teach fourth grade at a local school."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "Teaching fourth grade is compatible with being a teacher."
        ),
    },
    {
        "example_id": "synthetic_n03_location",
        "sanity_category": "location",
        "persona_text": (
            "- i live in paris.\n"
            "- i like jazz.\n"
            "- i own a bicycle."
        ),
        "dialogue_text": (
            "PARTNER: where do you live?\n"
            "SELF: i live in paris."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "SELF directly confirms the persona location."
        ),
    },
    {
        "example_id": "synthetic_n04_age",
        "sanity_category": "age",
        "persona_text": (
            "- i am 25 years old.\n"
            "- i enjoy cooking.\n"
            "- i have a cat."
        ),
        "dialogue_text": (
            "PARTNER: how old are you?\n"
            "SELF: i am 25 years old."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "SELF directly confirms the persona age."
        ),
    },
    {
        "example_id": "synthetic_n05_missing_evidence",
        "sanity_category": "missing_evidence",
        "persona_text": (
            "- i have a cat.\n"
            "- i work in finance.\n"
            "- i like apples."
        ),
        "dialogue_text": (
            "PARTNER: what do you do in your free time?\n"
            "SELF: i enjoy painting landscapes."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "The dialogue adds unrelated information and contradicts "
            "none of the persona statements."
        ),
    },
    {
        "example_id": "synthetic_n06_partner_distractor",
        "sanity_category": "partner_distractor",
        "persona_text": (
            "- i have no pets.\n"
            "- i enjoy reading.\n"
            "- i live in prague."
        ),
        "dialogue_text": (
            "PARTNER: i have two dogs and a cat.\n"
            "SELF: that is nice. i enjoy reading in my free time."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "The pet statement is made by PARTNER, not SELF."
        ),
    },
    {
        "example_id": "synthetic_n07_additional_information",
        "sanity_category": "additional_information",
        "persona_text": (
            "- i have two dogs.\n"
            "- i like music.\n"
            "- i work from home."
        ),
        "dialogue_text": (
            "PARTNER: do you have any other pets?\n"
            "SELF: yes, i also have a cat."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "Having a cat does not contradict having two dogs."
        ),
    },
    {
        "example_id": "synthetic_n08_temporal",
        "sanity_category": "temporal",
        "persona_text": (
            "- i live in london.\n"
            "- i like tea.\n"
            "- i have a brother."
        ),
        "dialogue_text": (
            "PARTNER: have you always lived in london?\n"
            "SELF: no, i lived in berlin as a child, "
            "but now i live in london."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "Past residence in Berlin is compatible with currently "
            "living in London."
        ),
    },
    {
        "example_id": "synthetic_n09_modal",
        "sanity_category": "modal",
        "persona_text": (
            "- i am a teacher.\n"
            "- i like hiking.\n"
            "- i own a car."
        ),
        "dialogue_text": (
            "PARTNER: do you have any plans for the future?\n"
            "SELF: i want to become a doctor someday."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "A future aspiration does not contradict the current profession."
        ),
    },
    {
        "example_id": "synthetic_n10_favorite",
        "sanity_category": "favorite",
        "persona_text": (
            "- my favorite color is blue.\n"
            "- i enjoy swimming.\n"
            "- i have a sister."
        ),
        "dialogue_text": (
            "PARTNER: do you like red?\n"
            "SELF: yes, i like red too, but blue is my favorite color."
        ),
        "expected_label": "NO_CONTRADICTION",
        "expected_label_binary": 0,
        "rationale": (
            "Liking red is compatible with blue being the favorite."
        ),
    },
]


synthetic_sanity_df = pd.DataFrame(
    synthetic_examples
)


# The inference script expects these metadata columns.
synthetic_sanity_df["pair_id"] = (
    synthetic_sanity_df[
        "example_id"
    ]
)

synthetic_sanity_df["dialogue_id"] = (
    synthetic_sanity_df[
        "example_id"
    ]
)

synthetic_sanity_df["variant"] = (
    "synthetic"
)


synthetic_sanity_df = synthetic_sanity_df[
    [
        "example_id",
        "pair_id",
        "dialogue_id",
        "variant",
        "persona_text",
        "dialogue_text",
        "expected_label",
        "expected_label_binary",
        "sanity_category",
        "rationale",
    ]
]


assert len(
    synthetic_sanity_df
) == 20

assert (
    synthetic_sanity_df[
        "expected_label_binary"
    ]
    .value_counts()
    .to_dict()
    == {
        1: 10,
        0: 10,
    }
)

assert (
    synthetic_sanity_df[
        "example_id"
    ].is_unique
)


synthetic_sanity_df.to_parquet(
    SYNTHETIC_SANITY_DATA_PATH,
    index=False,
)


print(
    f"Saved {len(synthetic_sanity_df)} "
    f"synthetic sanity examples to "
    f"{SYNTHETIC_SANITY_DATA_PATH}"
)

print()

print(
    synthetic_sanity_df[
        "expected_label"
    ].value_counts()
)

display(
    synthetic_sanity_df[
        [
            "example_id",
            "sanity_category",
            "expected_label",
        ]
    ]
)

Saved 20 synthetic sanity examples to data/llm_judge_synthetic_sanity_v1.parquet

expected_label
CONTRADICTION       10
NO_CONTRADICTION    10
Name: count, dtype: int64


,example_id,sanity_category,expected_label
0,synthetic_c01_preference,preference,CONTRADICTION
1,synthetic_c02_job,profession,CONTRADICTION
2,synthetic_c03_location,location,CONTRADICTION
3,synthetic_c04_age,age,CONTRADICTION
4,synthetic_c05_relationship,relationship_status,CONTRADICTION
5,synthetic_c06_pets,pets,CONTRADICTION
6,synthetic_c07_children,family,CONTRADICTION
7,synthetic_c08_diet,diet,CONTRADICTION
8,synthetic_c09_favorite,favorite,CONTRADICTION
9,synthetic_c10_activity,preference,CONTRADICTION


In [81]:
SYNTHETIC_JUDGE_CONFIGS = {
    "qwen3_4b": {
        "label": "Qwen3-4B",
        "file": (
            "qwen3_4b_"
            "synthetic_sanity_v1.parquet"
        ),
    },
    "phi4_mini": {
        "label": "Phi-4-mini-instruct",
        "file": (
            "phi4_mini_"
            "synthetic_sanity_v1.parquet"
        ),
    },
    "llama32_3b": {
        "label": "Llama-3.2-3B-Instruct",
        "file": (
            "llama32_3b_"
            "synthetic_sanity_v1.parquet"
        ),
    },
    "mistral7b": {
        "label": "Mistral-7B-Instruct-v0.3",
        "file": (
            "mistral7b_"
            "synthetic_sanity_v1.parquet"
        ),
    },
}


synthetic_judges = {
    model_id: load_judge_results(
        JUDGMENTS_DIR
        / config["file"]
    )
    for model_id, config
    in SYNTHETIC_JUDGE_CONFIGS.items()
}


expected_example_ids = set(
    synthetic_sanity_df[
        "example_id"
    ]
)


for model_id, df in (
    synthetic_judges.items()
):
    actual_example_ids = set(
        df[
            "example_id"
        ]
    )

    assert (
        actual_example_ids
        == expected_example_ids
    ), (
        f"{model_id}: synthetic "
        "example IDs do not match."
    )


print(
    "Loaded synthetic results "
    f"for {len(synthetic_judges)} judges."
)

Loaded synthetic results for 4 judges.


In [82]:
def summarize_synthetic_judge(
    judgments: pd.DataFrame,
    model_id: str,
) -> dict:
    valid = judgments[
        judgments[
            "valid_output"
        ]
    ].copy()

    valid[
        "correct"
    ] = (
        valid[
            "parsed_label_binary"
        ]
        == valid[
            "expected_label_binary"
        ]
    )

    contradiction_examples = (
        valid[
            valid[
                "expected_label_binary"
            ].eq(1)
        ]
    )

    no_contradiction_examples = (
        valid[
            valid[
                "expected_label_binary"
            ].eq(0)
        ]
    )

    return {
        "judge": (
            SYNTHETIC_JUDGE_CONFIGS[
                model_id
            ]["label"]
        ),
        "valid_output_rate": (
            judgments[
                "valid_output"
            ].mean()
        ),
        "accuracy": (
            valid[
                "correct"
            ].mean()
        ),
        "contradiction_recall": (
            contradiction_examples[
                "correct"
            ].mean()
        ),
        "no_contradiction_recall": (
            no_contradiction_examples[
                "correct"
            ].mean()
        ),
        "predicted_C_rate": (
            valid[
                "parsed_label_binary"
            ].mean()
        ),
    }


synthetic_summary = pd.DataFrame(
    [
        summarize_synthetic_judge(
            judgments,
            model_id,
        )
        for model_id, judgments
        in synthetic_judges.items()
    ]
).set_index(
    "judge"
)


display(
    synthetic_summary.style.format(
        "{:.1%}"
    )
)

,valid_output_rate,accuracy,contradiction_recall,no_contradiction_recall,predicted_C_rate
judge,,,,,
Qwen3-4B,100.0%,95.0%,100.0%,90.0%,55.0%
Phi-4-mini-instruct,100.0%,100.0%,100.0%,100.0%,50.0%
Llama-3.2-3B-Instruct,100.0%,95.0%,100.0%,90.0%,55.0%
Mistral-7B-Instruct-v0.3,100.0%,85.0%,100.0%,70.0%,65.0%


In [83]:
synthetic_prediction_matrix = (
    synthetic_sanity_df[
        [
            "example_id",
            "sanity_category",
            "expected_label",
            "expected_label_binary",
            "rationale",
        ]
    ]
    .set_index(
        "example_id"
    )
)


for model_id, judgments in (
    synthetic_judges.items()
):
    label = (
        SYNTHETIC_JUDGE_CONFIGS[
            model_id
        ]["label"]
    )

    predictions = (
        judgments
        .set_index(
            "example_id"
        )[
            "parsed_label"
        ]
    )

    synthetic_prediction_matrix[
        label
    ] = predictions


synthetic_prediction_matrix[
    "num_correct"
] = 0


judge_labels = [
    config["label"]
    for config
    in SYNTHETIC_JUDGE_CONFIGS.values()
]


for label in judge_labels:
    synthetic_prediction_matrix[
        "num_correct"
    ] += (
        synthetic_prediction_matrix[
            label
        ]
        == synthetic_prediction_matrix[
            "expected_label"
        ]
    ).astype(int)


synthetic_prediction_matrix[
    "all_correct"
] = (
    synthetic_prediction_matrix[
        "num_correct"
    ]
    == len(
        judge_labels
    )
)


display(
    synthetic_prediction_matrix[
        [
            "sanity_category",
            "expected_label",
            *judge_labels,
            "num_correct",
            "rationale",
        ]
    ]
    .sort_values(
        [
            "num_correct",
            "expected_label",
        ]
    )
)

,sanity_category,expected_label,Qwen3-4B,Phi-4-mini-instruct,Llama-3.2-3B-Instruct,Mistral-7B-Instruct-v0.3,num_correct,rationale
example_id,,,,,,,,
synthetic_n07_additional_information,additional_information,NO_CONTRADICTION,CONTRADICTION,NO_CONTRADICTION,NO_CONTRADICTION,CONTRADICTION,2,Having a cat does not contradict having two dogs.
synthetic_n10_favorite,favorite,NO_CONTRADICTION,NO_CONTRADICTION,NO_CONTRADICTION,CONTRADICTION,CONTRADICTION,2,Liking red is compatible with blue being the f...
synthetic_n09_modal,modal,NO_CONTRADICTION,NO_CONTRADICTION,NO_CONTRADICTION,NO_CONTRADICTION,CONTRADICTION,3,A future aspiration does not contradict the cu...
synthetic_c01_preference,preference,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,SELF explicitly says they love chocolate while...
synthetic_c02_job,profession,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,SELF explicitly denies currently working as a ...
synthetic_c03_location,location,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,SELF explicitly denies living in the persona l...
synthetic_c04_age,age,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,The current ages are mutually incompatible.
synthetic_c05_relationship,relationship_status,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,SELF explicitly says they are married while th...
synthetic_c06_pets,pets,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,CONTRADICTION,4,Having two dogs contradicts having no pets.


### 3.3 Pilot judge analysis

Before running the full experiment, each LLM judge is evaluated on the same
pilot sample of matched original and augmented persona–dialogue pairs.

For each judge, the analysis reports:

- output validity,
- distribution of predicted labels,
- predictions separately for original and augmented variants,
- agreement with the automatically derived expected labels,
- matched-pair prediction transitions,
- contradiction rates,
- examples resembling false negatives and false positives.

The expected labels are used as an experimental reference rather than manually
verified ground truth.

In [40]:
def load_judge_results(
    path: str | Path,
    expected_prompt_id: str = PROMPT_ID,
) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()

    prompt_ids = df[
        "prompt_id"
    ].dropna().unique()

    if (
        len(prompt_ids) != 1
        or prompt_ids[0]
        != expected_prompt_id
    ):
        raise ValueError(
            f"{path}: expected prompt "
            f"{expected_prompt_id!r}, "
            f"found {prompt_ids}."
        )

    parsed = df[
        "raw_output"
    ].map(parse_judge_output)

    df["parsed_label"] = parsed.map(
        lambda result: result[0]
    )

    df["parsed_label_binary"] = parsed.map(
        lambda result: result[1]
    )

    df["valid_output"] = (
        df["parsed_label"].notna()
    )

    return df

def analyze_judge_results(
    df: pd.DataFrame,
    model_label: str,
) -> dict:
    print("=" * 70)
    print(model_label)
    print("=" * 70)

    print(f"Rows: {len(df):,}")
    print(
        f"Pairs: "
        f"{df['pair_id'].nunique():,}"
    )
    print(
        "Valid output rate:",
        f"{df['valid_output'].mean():.1%}",
    )

    print("\nParsed labels:")
    label_counts = (
        df["parsed_label"]
        .value_counts(dropna=False)
    )
    print(label_counts)

    print("\nPredictions by variant:")
    variant_predictions = pd.crosstab(
        df["variant"],
        df["parsed_label"],
        dropna=False,
    )
    display(variant_predictions)

    valid = df[
        df["valid_output"]
    ].copy()

    valid["matches_expected"] = (
        valid["parsed_label_binary"]
        == valid["expected_label_binary"]
    )

    expected_agreement = (
        valid["matches_expected"].mean()
    )

    print(
        "\nAgreement with expected labels:",
        f"{expected_agreement:.1%}",
    )

    agreement_by_variant = (
        valid
        .groupby("variant")[
            "matches_expected"
        ]
        .mean()
    )

    print(
        "\nAgreement with expected labels "
        "by variant:"
    )
    print(
        agreement_by_variant.map(
            lambda value: f"{value:.1%}"
        )
    )

    contradiction_rate = (
        valid[
            "parsed_label_binary"
        ].mean()
    )

    print(
        "\nOverall contradiction rate:",
        f"{contradiction_rate:.1%}",
    )

    contradiction_by_variant = (
        valid
        .groupby("variant")[
            "parsed_label_binary"
        ]
        .mean()
    )

    print(
        "\nContradiction rate by variant:"
    )
    print(
        contradiction_by_variant.map(
            lambda value: f"{value:.1%}"
        )
    )

    pair_predictions = (
        valid
        .pivot(
            index="pair_id",
            columns="variant",
            values="parsed_label",
        )
        .dropna(
            subset=[
                "original",
                "augmented",
            ]
        )
        .copy()
    )

    pair_predictions["transition"] = (
        pair_predictions["original"]
        + " → "
        + pair_predictions["augmented"]
    )

    transition_counts = (
        pair_predictions[
            "transition"
        ].value_counts()
    )

    print(
        "\nMatched-pair transitions:"
    )
    print(transition_counts)

    ideal_transition_rate = (
        pair_predictions[
            "transition"
        ]
        .eq(
            "NO_CONTRADICTION → CONTRADICTION"
        )
        .mean()
    )

    print(
        "\nExpected N → C transition rate:",
        f"{ideal_transition_rate:.1%}",
    )

    invalid_outputs = df[
        ~df["valid_output"]
    ][
        [
            "example_id",
            "variant",
            "raw_output",
        ]
    ].copy()

    if not invalid_outputs.empty:
        print(
            f"\nInvalid outputs: "
            f"{len(invalid_outputs)}"
        )
        display(invalid_outputs)

    return {
        "valid": valid,
        "label_counts": label_counts,
        "variant_predictions": variant_predictions,
        "agreement_by_variant": agreement_by_variant,
        "contradiction_by_variant": contradiction_by_variant,
        "pair_predictions": pair_predictions,
        "transition_counts": transition_counts,
        "invalid_outputs": invalid_outputs,
    }

def get_judge_error_examples(
    judgments_df: pd.DataFrame,
    experiment_df: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    Return augmented examples predicted as NO_CONTRADICTION and
    original examples predicted as CONTRADICTION.

    These resemble false negatives and false positives relative to the
    automatically derived expected labels.
    """
    inspection = judgments_df.merge(
        experiment_df[
            [
                "example_id",
                "persona_text",
                "dialogue_text",
                "changed_original_facts",
                "changed_augmented_facts",
                "changed_grounding_labels",
            ]
        ],
        on="example_id",
        how="left",
    )

    false_negative_like = inspection[
        inspection["variant"].eq(
            "augmented"
        )
        & inspection[
            "parsed_label"
        ].eq(
            "NO_CONTRADICTION"
        )
    ].copy()

    false_positive_like = inspection[
        inspection["variant"].eq(
            "original"
        )
        & inspection[
            "parsed_label"
        ].eq(
            "CONTRADICTION"
        )
    ].copy()

    return (
        false_negative_like,
        false_positive_like,
    )

In [66]:
JUDGE_CONFIGS = {
    "qwen3_4b": {
        "label": "Qwen3-4B",
        "file": "qwen3_4b_pilot_v3.parquet",
    },
    "phi4_mini": {
        "label": "Phi-4-mini-instruct",
        "file": "phi4_mini_pilot_v3.parquet",
    },
    "llama32_3b": {
        "label": "Llama-3.2-3B-Instruct",
        "file": "llama32_3b_pilot_v3.parquet",
    },
    "mistral7b": {
        "label": "Mistral-7B-Instruct-v0.3",
        "file": "mistral7b_pilot_v3.parquet",
    },
    "granite33_2b": {
    "label": "Granite-3.3-2B-Instruct",
    "file": "granite33_2b_pilot_v3.parquet",
    },
}

def run_pilot_analysis(
    model_id: str,
    example_limit: int = 5,
):
    """
    Run pilot analysis for one judge and print the complete summary
    as a single copyable plain-text block.
    """
    config = JUDGE_CONFIGS[
        model_id
    ]

    judgments = load_judge_results(
        JUDGMENTS_DIR
        / config["file"]
    )

    valid = judgments[
        judgments["valid_output"]
    ].copy()

    valid["matches_expected"] = (
        valid["parsed_label_binary"]
        == valid["expected_label_binary"]
    )

    # ---------------------------------------------------------
    # Basic summaries
    # ---------------------------------------------------------

    label_counts = (
        judgments[
            "parsed_label"
        ]
        .value_counts(
            dropna=False
        )
    )

    variant_predictions = pd.crosstab(
        judgments["variant"],
        judgments["parsed_label"],
        dropna=False,
    )

    agreement_with_expected = (
        valid[
            "matches_expected"
        ].mean()
    )

    agreement_by_variant = (
        valid
        .groupby("variant")[
            "matches_expected"
        ]
        .mean()
    )

    contradiction_rate = (
        valid[
            "parsed_label_binary"
        ].mean()
    )

    contradiction_by_variant = (
        valid
        .groupby("variant")[
            "parsed_label_binary"
        ]
        .mean()
    )

    # ---------------------------------------------------------
    # Matched-pair analysis
    # ---------------------------------------------------------

    pair_predictions = (
        valid
        .pivot(
            index="pair_id",
            columns="variant",
            values="parsed_label",
        )
    )

    if (
        "original" in pair_predictions.columns
        and "augmented" in pair_predictions.columns
    ):
        pair_predictions = (
            pair_predictions
            .dropna(
                subset=[
                    "original",
                    "augmented",
                ]
            )
            .copy()
        )

        pair_predictions[
            "transition"
        ] = (
            pair_predictions["original"]
            + " → "
            + pair_predictions["augmented"]
        )

        transition_counts = (
            pair_predictions[
                "transition"
            ]
            .value_counts()
        )

        expected_transition_rate = (
            pair_predictions[
                "transition"
            ]
            .eq(
                "NO_CONTRADICTION → CONTRADICTION"
            )
            .mean()
        )
    else:
        pair_predictions = (
            pd.DataFrame()
        )

        transition_counts = (
            pd.Series(
                dtype=int
            )
        )

        expected_transition_rate = float("nan")

    # ---------------------------------------------------------
    # Error-like examples
    # ---------------------------------------------------------

    (
        false_negative_like,
        false_positive_like,
    ) = get_judge_error_examples(
        judgments,
        experiment_df,
    )

    invalid_outputs = judgments[
        ~judgments[
            "valid_output"
        ]
    ].copy()

    # ---------------------------------------------------------
    # Collect analysis results
    # ---------------------------------------------------------

    analysis = {
        "valid": valid,
        "label_counts": label_counts,
        "variant_predictions": (
            variant_predictions
        ),
        "agreement_with_expected": (
            agreement_with_expected
        ),
        "agreement_by_variant": (
            agreement_by_variant
        ),
        "contradiction_rate": (
            contradiction_rate
        ),
        "contradiction_by_variant": (
            contradiction_by_variant
        ),
        "pair_predictions": (
            pair_predictions
        ),
        "transition_counts": (
            transition_counts
        ),
        "expected_transition_rate": (
            expected_transition_rate
        ),
        "invalid_outputs": (
            invalid_outputs
        ),
    }

    # ---------------------------------------------------------
    # Build one copyable text block
    # ---------------------------------------------------------

    lines = []

    lines.append("=" * 70)
    lines.append(config["label"])
    lines.append("=" * 70)

    lines.append(
        f"Model ID: {model_id}"
    )

    lines.append(
        f"Prompt ID: "
        f"{judgments['prompt_id'].iloc[0]}"
    )

    if "model_revision" in judgments.columns:
        lines.append(
            f"Model revision: "
            f"{judgments['model_revision'].iloc[0]}"
        )

    lines.append(
        f"Rows: {len(judgments):,}"
    )

    lines.append(
        f"Pairs: "
        f"{judgments['pair_id'].nunique():,}"
    )

    lines.append(
        "Valid output rate: "
        f"{judgments['valid_output'].mean():.1%}"
    )

    lines.append(
        f"Invalid outputs: "
        f"{len(invalid_outputs):,}"
    )

    lines.append(
        "\nParsed labels:"
    )
    lines.append(
        label_counts.to_string()
    )

    lines.append(
        "\nPredictions by variant:"
    )
    lines.append(
        variant_predictions.to_string()
    )

    lines.append(
        "\nAgreement with expected labels: "
        f"{agreement_with_expected:.1%}"
    )

    lines.append(
        "\nAgreement with expected labels "
        "by variant:"
    )

    for variant, value in (
        agreement_by_variant.items()
    ):
        lines.append(
            f"  {variant}: {value:.1%}"
        )

    lines.append(
        "\nOverall contradiction rate: "
        f"{contradiction_rate:.1%}"
    )

    lines.append(
        "\nContradiction rate by variant:"
    )

    for variant, value in (
        contradiction_by_variant.items()
    ):
        lines.append(
            f"  {variant}: {value:.1%}"
        )

    lines.append(
        "\nMatched-pair transitions:"
    )

    if len(transition_counts):
        for transition, count in (
            transition_counts.items()
        ):
            lines.append(
                f"  {transition}: {count}"
            )
    else:
        lines.append(
            "  No complete valid pairs."
        )

    if pd.notna(
        expected_transition_rate
    ):
        lines.append(
            "\nExpected N → C transition rate: "
            f"{expected_transition_rate:.1%}"
        )

    lines.append(
        "\nError-like counts:"
    )

    lines.append(
        "  Augmented → NO_CONTRADICTION: "
        f"{len(false_negative_like)}"
    )

    lines.append(
        "  Original → CONTRADICTION: "
        f"{len(false_positive_like)}"
    )

    # ---------------------------------------------------------
    # Selected examples
    # ---------------------------------------------------------

    if (
        example_limit > 0
        and not false_negative_like.empty
    ):
        lines.append(
            "\nSample augmented → "
            "NO_CONTRADICTION examples:"
        )

        for _, row in (
            false_negative_like
            .head(example_limit)
            .iterrows()
        ):
            lines.append(
                "\n" + "-" * 60
            )

            lines.append(
                f"Example: "
                f"{row['example_id']}"
            )

            lines.append(
                "Changed original facts: "
                f"{row['changed_original_facts']}"
            )

            lines.append(
                "Changed augmented facts: "
                f"{row['changed_augmented_facts']}"
            )

            lines.append(
                "Persona:"
            )
            lines.append(
                str(row["persona_text"])
            )

            lines.append(
                "Dialogue:"
            )
            lines.append(
                str(row["dialogue_text"])
            )

            lines.append(
                "Raw output: "
                f"{row['raw_output']}"
            )

    if (
        example_limit > 0
        and not false_positive_like.empty
    ):
        lines.append(
            "\nSample original → "
            "CONTRADICTION examples:"
        )

        for _, row in (
            false_positive_like
            .head(example_limit)
            .iterrows()
        ):
            lines.append(
                "\n" + "-" * 60
            )

            lines.append(
                f"Example: "
                f"{row['example_id']}"
            )

            lines.append(
                "Persona:"
            )
            lines.append(
                str(row["persona_text"])
            )

            lines.append(
                "Dialogue:"
            )
            lines.append(
                str(row["dialogue_text"])
            )

            lines.append(
                "Raw output: "
                f"{row['raw_output']}"
            )

    # A single print = one PyCharm notebook output.
    print(
        "\n".join(lines)
    )

    return (
        judgments,
        analysis,
        false_negative_like,
        false_positive_like,
    )

In [67]:
(
    qwen_pilot,
    qwen_analysis,
    qwen_false_negative_like,
    qwen_false_positive_like,
) = run_pilot_analysis(
    "qwen3_4b"
)

Qwen3-4B
Model ID: qwen3_4b
Prompt ID: full_persona_v3
Model revision: 1cfa9a7208912126459214e8b04321603b3df60c
Rows: 100
Pairs: 50
Valid output rate: 100.0%
Invalid outputs: 0

Parsed labels:
parsed_label
NO_CONTRADICTION    74
CONTRADICTION       26

Predictions by variant:
parsed_label  CONTRADICTION  NO_CONTRADICTION
variant                                      
augmented                19                31
original                  7                43

Agreement with expected labels: 62.0%

Agreement with expected labels by variant:
  augmented: 38.0%
  original: 86.0%

Overall contradiction rate: 26.0%

Contradiction rate by variant:
  augmented: 38.0%
  original: 14.0%

Matched-pair transitions:
  NO_CONTRADICTION → NO_CONTRADICTION: 28
  NO_CONTRADICTION → CONTRADICTION: 15
  CONTRADICTION → CONTRADICTION: 4
  CONTRADICTION → NO_CONTRADICTION: 3

Expected N → C transition rate: 30.0%

Error-like counts:
  Augmented → NO_CONTRADICTION: 31
  Original → CONTRADICTION: 7

Sample au

In [68]:
(
    phi_pilot,
    phi_analysis,
    phi_false_negative_like,
    phi_false_positive_like,
) = run_pilot_analysis(
    "phi4_mini"
)

Phi-4-mini-instruct
Model ID: phi4_mini
Prompt ID: full_persona_v3
Model revision: cfbefacb99257ffa30c83adab238a50856ac3083
Rows: 100
Pairs: 50
Valid output rate: 100.0%
Invalid outputs: 0

Parsed labels:
parsed_label
NO_CONTRADICTION    91
CONTRADICTION        9

Predictions by variant:
parsed_label  CONTRADICTION  NO_CONTRADICTION
variant                                      
augmented                 9                41
original                  0                50

Agreement with expected labels: 59.0%

Agreement with expected labels by variant:
  augmented: 18.0%
  original: 100.0%

Overall contradiction rate: 9.0%

Contradiction rate by variant:
  augmented: 18.0%
  original: 0.0%

Matched-pair transitions:
  NO_CONTRADICTION → NO_CONTRADICTION: 41
  NO_CONTRADICTION → CONTRADICTION: 9

Expected N → C transition rate: 18.0%

Error-like counts:
  Augmented → NO_CONTRADICTION: 41
  Original → CONTRADICTION: 0

Sample augmented → NO_CONTRADICTION examples:

-------------------------

In [69]:
(
    llama_pilot,
    llama_analysis,
    llama_false_negative_like,
    llama_false_positive_like,
) = run_pilot_analysis(
    "llama32_3b"
)

Llama-3.2-3B-Instruct
Model ID: llama32_3b
Prompt ID: full_persona_v3
Model revision: 0cb88a4f764b7a12671c53f0838cd831a0843b95
Rows: 100
Pairs: 50
Valid output rate: 100.0%
Invalid outputs: 0

Parsed labels:
parsed_label
NO_CONTRADICTION    92
CONTRADICTION        8

Predictions by variant:
parsed_label  CONTRADICTION  NO_CONTRADICTION
variant                                      
augmented                 7                43
original                  1                49

Agreement with expected labels: 56.0%

Agreement with expected labels by variant:
  augmented: 14.0%
  original: 98.0%

Overall contradiction rate: 8.0%

Contradiction rate by variant:
  augmented: 14.0%
  original: 2.0%

Matched-pair transitions:
  NO_CONTRADICTION → NO_CONTRADICTION: 43
  NO_CONTRADICTION → CONTRADICTION: 6
  CONTRADICTION → CONTRADICTION: 1

Expected N → C transition rate: 12.0%

Error-like counts:
  Augmented → NO_CONTRADICTION: 43
  Original → CONTRADICTION: 1

Sample augmented → NO_CONTRADICTION

In [70]:
(
    mistral_pilot,
    mistral_analysis,
    mistral_false_negative_like,
    mistral_false_positive_like,
) = run_pilot_analysis(
    "mistral7b"
)

Mistral-7B-Instruct-v0.3
Model ID: mistral7b
Prompt ID: full_persona_v3
Model revision: c170c708c41dac9275d15a8fff4eca08d52bab71
Rows: 100
Pairs: 50
Valid output rate: 100.0%
Invalid outputs: 0

Parsed labels:
parsed_label
CONTRADICTION       53
NO_CONTRADICTION    47

Predictions by variant:
parsed_label  CONTRADICTION  NO_CONTRADICTION
variant                                      
augmented                36                14
original                 17                33

Agreement with expected labels: 69.0%

Agreement with expected labels by variant:
  augmented: 72.0%
  original: 66.0%

Overall contradiction rate: 53.0%

Contradiction rate by variant:
  augmented: 72.0%
  original: 34.0%

Matched-pair transitions:
  NO_CONTRADICTION → CONTRADICTION: 19
  CONTRADICTION → CONTRADICTION: 17
  NO_CONTRADICTION → NO_CONTRADICTION: 14

Expected N → C transition rate: 38.0%

Error-like counts:
  Augmented → NO_CONTRADICTION: 14
  Original → CONTRADICTION: 17

Sample augmented → NO_CONTRA

### 3.4 Inter-judge pilot comparison

After inspecting each judge individually, agreement is evaluated directly
between judges on the common pilot sample.

The pilot comparison is used to verify that all four judges produce usable
outputs and exhibit non-trivial decision behaviour before running the full
experiment.

The main pilot statistics are:

- raw pairwise agreement,
- Cohen's kappa,
- agreement separately for original and augmented variants,
- judge-specific contradiction rates.

### 3.4 Inter-judge pilot comparison

The pilot results are compared across all currently available judges.

The analysis is intentionally dynamic: it can be run with the three core
judges and automatically includes a fourth judge once either Llama 3.2 3B
or the Granite fallback result becomes available.

The main quantities reported here are:
- contradiction rates,
- pairwise raw agreement,
- pairwise Cohen's kappa,
- agreement separately for original and augmented examples,
- judge consensus and disagreement patterns.

In [72]:
from sklearn.metrics import (
    cohen_kappa_score,
)


CORE_PILOT_JUDGE_IDS = [
    "qwen3_4b",
    "phi4_mini",
    "mistral7b",
]

FOURTH_JUDGE_CANDIDATES = [
    "llama32_3b",
    "granite33_2b",
]


def result_file_exists(
    model_id: str,
) -> bool:
    return (
        JUDGMENTS_DIR
        / JUDGE_CONFIGS[
            model_id
        ]["file"]
    ).exists()


missing_core = [
    model_id
    for model_id
    in CORE_PILOT_JUDGE_IDS
    if not result_file_exists(
        model_id
    )
]

if missing_core:
    raise FileNotFoundError(
        "Missing core pilot results: "
        + ", ".join(missing_core)
    )


fourth_judge_id = next(
    (
        model_id
        for model_id
        in FOURTH_JUDGE_CANDIDATES
        if result_file_exists(
            model_id
        )
    ),
    None,
)


ACTIVE_PILOT_JUDGE_IDS = (
    CORE_PILOT_JUDGE_IDS.copy()
)

if fourth_judge_id is not None:
    ACTIVE_PILOT_JUDGE_IDS.append(
        fourth_judge_id
    )


print(
    "Active pilot judges:"
)

for model_id in (
    ACTIVE_PILOT_JUDGE_IDS
):
    print(
        f"  - "
        f"{JUDGE_CONFIGS[model_id]['label']}"
    )

Active pilot judges:
  - Qwen3-4B
  - Phi-4-mini-instruct
  - Mistral-7B-Instruct-v0.3
  - Llama-3.2-3B-Instruct


In [73]:
pilot_judges = {
    model_id: load_judge_results(
        JUDGMENTS_DIR
        / JUDGE_CONFIGS[
            model_id
        ]["file"]
    )
    for model_id
    in ACTIVE_PILOT_JUDGE_IDS
}


# Verify that all judges evaluated
# exactly the same examples.
reference_examples = set(
    next(
        iter(
            pilot_judges.values()
        )
    )["example_id"]
)

for model_id, df in (
    pilot_judges.items()
):
    assert (
        set(df["example_id"])
        == reference_examples
    ), (
        f"{model_id} did not evaluate "
        "the same pilot examples."
    )

print(
    f"Validated common sample: "
    f"{len(reference_examples)} examples."
)

Validated common sample: 100 examples.


In [74]:
def summarize_pilot_judge(
    judgments: pd.DataFrame,
    model_id: str,
) -> dict:
    valid = judgments[
        judgments["valid_output"]
    ].copy()

    original = valid[
        valid["variant"].eq(
            "original"
        )
    ]

    augmented = valid[
        valid["variant"].eq(
            "augmented"
        )
    ]

    expected_agreement = (
        valid[
            "parsed_label_binary"
        ]
        == valid[
            "expected_label_binary"
        ]
    ).mean()

    pair_predictions = (
        valid
        .pivot(
            index="pair_id",
            columns="variant",
            values="parsed_label_binary",
        )
        .dropna(
            subset=[
                "original",
                "augmented",
            ]
        )
    )

    ideal_transition = (
        (
            pair_predictions[
                "original"
            ].eq(0)
        )
        & (
            pair_predictions[
                "augmented"
            ].eq(1)
        )
    ).mean()

    original_rate = (
        original[
            "parsed_label_binary"
        ].mean()
    )

    augmented_rate = (
        augmented[
            "parsed_label_binary"
        ].mean()
    )

    return {
        "judge": (
            JUDGE_CONFIGS[
                model_id
            ]["label"]
        ),
        "valid_output_rate": (
            judgments[
                "valid_output"
            ].mean()
        ),
        "contradiction_rate": (
            valid[
                "parsed_label_binary"
            ].mean()
        ),
        "original_C_rate": (
            original_rate
        ),
        "augmented_C_rate": (
            augmented_rate
        ),
        "augmentation_shift": (
            augmented_rate
            - original_rate
        ),
        "expected_label_agreement": (
            expected_agreement
        ),
        "N_to_C_pair_rate": (
            ideal_transition
        ),
    }


pilot_summary = pd.DataFrame(
    [
        summarize_pilot_judge(
            pilot_judges[
                model_id
            ],
            model_id,
        )
        for model_id
        in ACTIVE_PILOT_JUDGE_IDS
    ]
).set_index(
    "judge"
)


pilot_summary.style.format(
    {
        column: "{:.1%}"
        for column
        in pilot_summary.columns
    }
)

,valid_output_rate,contradiction_rate,original_C_rate,augmented_C_rate,augmentation_shift,expected_label_agreement,N_to_C_pair_rate
judge,,,,,,,
Qwen3-4B,100.0%,26.0%,14.0%,38.0%,24.0%,62.0%,30.0%
Phi-4-mini-instruct,100.0%,9.0%,0.0%,18.0%,18.0%,59.0%,18.0%
Mistral-7B-Instruct-v0.3,100.0%,53.0%,34.0%,72.0%,38.0%,69.0%,38.0%
Llama-3.2-3B-Instruct,100.0%,8.0%,2.0%,14.0%,12.0%,56.0%,12.0%


#### Pilot results by augmentation type

The augmented pilot examples are additionally grouped by the type of
rule-based augmentation applied to the persona.

Because a profile may contain more than one augmentation type, two descriptive
summaries are reported:

1. **Single-type profiles** – profiles containing only one augmentation type.
   This provides the cleanest comparison between augmentation categories.
2. **Profiles containing each type** – all profiles in which a given
   augmentation type occurs. Multi-type profiles may therefore contribute to
   more than one category in this second summary.

These summaries are descriptive pilot diagnostics and are not used as the
main measure of judge performance.

In [86]:
def normalize_list_value(
    value,
) -> list:
    """
    Convert parquet list-like values to a regular Python list.
    """
    if value is None:
        return []

    if isinstance(
        value,
        str,
    ):
        return [
            value
        ]

    if hasattr(
        value,
        "tolist",
    ):
        value = value.tolist()

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            item
            for item in value
            if item is not None
        ]

    return [
        value
    ]


def unique_string_values(
    value,
) -> list[str]:
    """
    Normalize a list-like value and remove duplicates
    while preserving order.
    """
    values = [
        str(item)
        for item
        in normalize_list_value(
            value
        )
    ]

    return list(
        dict.fromkeys(
            values
        )
    )


if (
    "augmentation_types"
    not in pilot_df.columns
):
    raise KeyError(
        "Expected column "
        "'augmentation_types' "
        "is missing from pilot_df. "
        "Available augmentation-related "
        "columns are: "
        + str(
            [
                column
                for column
                in pilot_df.columns
                if "augmentation"
                in column
            ]
        )
    )


pilot_augmented_metadata = (
    pilot_df[
        pilot_df[
            "variant"
        ].eq(
            "augmented"
        )
    ][
        [
            "example_id",
            "pair_id",
            "augmentation_types",
            "pair_num_augmented_facts",
            "pair_num_grounded_augmented_facts",
        ]
    ]
    .drop_duplicates(
        "example_id"
    )
    .copy()
)


pilot_augmented_metadata[
    "augmentation_types_clean"
] = (
    pilot_augmented_metadata[
        "augmentation_types"
    ]
    .map(
        unique_string_values
    )
)


pilot_augmented_metadata[
    "num_augmentation_types"
] = (
    pilot_augmented_metadata[
        "augmentation_types_clean"
    ]
    .map(
        len
    )
)


pilot_augmented_metadata[
    "augmentation_type_combination"
] = (
    pilot_augmented_metadata[
        "augmentation_types_clean"
    ]
    .map(
        lambda values:
            " + ".join(
                values
            )
    )
)


print(
    "Augmented pilot profiles:",
    len(
        pilot_augmented_metadata
    ),
)

print(
    "\nNumber of distinct "
    "augmentation types per profile:"
)

display(
    pilot_augmented_metadata[
        "num_augmentation_types"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "profiles"
    )
)

Augmented pilot profiles: 50

Number of distinct augmentation types per profile:


,profiles
num_augmentation_types,
1,34
2,13
3,3


In [87]:
augmentation_judge_rows = []


for model_id, judgments in (
    pilot_judges.items()
):
    judge_predictions = (
        judgments[
            judgments[
                "variant"
            ].eq(
                "augmented"
            )
        ][
            [
                "example_id",
                "parsed_label_binary",
                "valid_output",
            ]
        ]
        .merge(
            pilot_augmented_metadata,
            on="example_id",
            how="inner",
        )
        .copy()
    )

    judge_predictions[
        "judge"
    ] = (
        JUDGE_CONFIGS[
            model_id
        ]["label"]
    )

    augmentation_judge_rows.append(
        judge_predictions
    )


augmentation_judge_df = pd.concat(
    augmentation_judge_rows,
    ignore_index=True,
)


assert (
    augmentation_judge_df[
        "judge"
    ].nunique()
    == len(
        pilot_judges
    )
)


print(
    "Judge-example rows:",
    len(
        augmentation_judge_df
    ),
)

Judge-example rows: 200


In [88]:
single_type_judgments = (
    augmentation_judge_df[
        augmentation_judge_df[
            "valid_output"
        ]
        & augmentation_judge_df[
            "num_augmentation_types"
        ].eq(1)
    ]
    .copy()
)


single_type_judgments[
    "augmentation_type"
] = (
    single_type_judgments[
        "augmentation_types_clean"
    ]
    .map(
        lambda values:
            values[0]
    )
)


single_type_summary = (
    single_type_judgments
    .groupby(
        [
            "augmentation_type",
            "judge",
        ],
        observed=True,
    )
    .agg(
        examples=(
            "example_id",
            "nunique",
        ),
        contradiction_rate=(
            "parsed_label_binary",
            "mean",
        ),
    )
    .reset_index()
)


single_type_rates = (
    single_type_summary
    .pivot(
        index="augmentation_type",
        columns="judge",
        values="contradiction_rate",
    )
)


single_type_counts = (
    single_type_summary
    .pivot(
        index="augmentation_type",
        columns="judge",
        values="examples",
    )
)


print(
    "Single-type profiles — "
    "contradiction rate:"
)

display(
    single_type_rates.style.format(
        "{:.1%}"
    )
)


print(
    "Single-type profiles — "
    "number of examples:"
)

display(
    single_type_counts
)

Single-type profiles — contradiction rate:


judge,Llama-3.2-3B-Instruct,Mistral-7B-Instruct-v0.3,Phi-4-mini-instruct,Qwen3-4B
augmentation_type,,,,
age_swap,0.0%,100.0%,33.3%,33.3%
aspiration_negation,50.0%,100.0%,0.0%,50.0%
count_swap,0.0%,100.0%,0.0%,0.0%
diet_swap,0.0%,100.0%,0.0%,0.0%
family_composition_swap,0.0%,0.0%,0.0%,0.0%
favorite_swap,0.0%,20.0%,0.0%,20.0%
location_swap,0.0%,0.0%,0.0%,0.0%
preference_negation,17.6%,70.6%,5.9%,35.3%
profession_swap,0.0%,50.0%,0.0%,50.0%


Single-type profiles — number of examples:


judge,Llama-3.2-3B-Instruct,Mistral-7B-Instruct-v0.3,Phi-4-mini-instruct,Qwen3-4B
augmentation_type,,,,
age_swap,3,3,3,3
aspiration_negation,2,2,2,2
count_swap,2,2,2,2
diet_swap,1,1,1,1
family_composition_swap,1,1,1,1
favorite_swap,5,5,5,5
location_swap,1,1,1,1
preference_negation,17,17,17,17
profession_swap,2,2,2,2


In [89]:
all_type_judgments = (
    augmentation_judge_df[
        augmentation_judge_df[
            "valid_output"
        ]
    ]
    .copy()
)


all_type_judgments[
    "augmentation_type"
] = (
    all_type_judgments[
        "augmentation_types_clean"
    ]
)


all_type_judgments = (
    all_type_judgments
    .explode(
        "augmentation_type"
    )
    .drop_duplicates(
        [
            "judge",
            "example_id",
            "augmentation_type",
        ]
    )
)


all_type_summary = (
    all_type_judgments
    .groupby(
        [
            "augmentation_type",
            "judge",
        ],
        observed=True,
    )
    .agg(
        examples=(
            "example_id",
            "nunique",
        ),
        contradiction_rate=(
            "parsed_label_binary",
            "mean",
        ),
    )
    .reset_index()
)


all_type_rates = (
    all_type_summary
    .pivot(
        index="augmentation_type",
        columns="judge",
        values="contradiction_rate",
    )
)


all_type_counts = (
    all_type_summary
    .pivot(
        index="augmentation_type",
        columns="judge",
        values="examples",
    )
)


print(
    "All profiles containing each "
    "augmentation type — "
    "contradiction rate:"
)

display(
    all_type_rates.style.format(
        "{:.1%}"
    )
)


print(
    "All profiles containing each "
    "augmentation type — "
    "number of examples:"
)

display(
    all_type_counts
)

All profiles containing each augmentation type — contradiction rate:


judge,Llama-3.2-3B-Instruct,Mistral-7B-Instruct-v0.3,Phi-4-mini-instruct,Qwen3-4B
augmentation_type,,,,
age_swap,0.0%,100.0%,25.0%,25.0%
aspiration_negation,50.0%,100.0%,0.0%,50.0%
count_swap,0.0%,75.0%,0.0%,0.0%
diet_swap,0.0%,100.0%,50.0%,0.0%
family_composition_swap,0.0%,60.0%,20.0%,60.0%
favorite_swap,18.2%,54.5%,36.4%,45.5%
living_situation_swap,0.0%,100.0%,50.0%,100.0%
location_swap,0.0%,75.0%,0.0%,25.0%
preference_negation,21.4%,78.6%,25.0%,50.0%


All profiles containing each augmentation type — number of examples:


judge,Llama-3.2-3B-Instruct,Mistral-7B-Instruct-v0.3,Phi-4-mini-instruct,Qwen3-4B
augmentation_type,,,,
age_swap,4,4,4,4
aspiration_negation,2,2,2,2
count_swap,4,4,4,4
diet_swap,2,2,2,2
family_composition_swap,5,5,5,5
favorite_swap,11,11,11,11
living_situation_swap,2,2,2,2
location_swap,4,4,4,4
preference_negation,28,28,28,28


In [90]:
pilot_augmented_predictions = (
    pilot_augmented_metadata[
        [
            "example_id",
            "pair_id",
            "augmentation_types_clean",
        ]
    ]
    .set_index("example_id")
)


for model_id, judgments in pilot_judges.items():
    label = JUDGE_CONFIGS[
        model_id
    ]["label"]

    predictions = (
        judgments[
            judgments["variant"].eq(
                "augmented"
            )
        ]
        .set_index("example_id")[
            "parsed_label_binary"
        ]
    )

    pilot_augmented_predictions[
        label
    ] = predictions


judge_labels = [
    JUDGE_CONFIGS[
        model_id
    ]["label"]
    for model_id
    in ACTIVE_PILOT_JUDGE_IDS
]


pilot_augmented_predictions[
    "contradiction_votes"
] = (
    pilot_augmented_predictions[
        judge_labels
    ]
    .sum(axis=1)
)


augmentation_inspection = (
    pilot_augmented_predictions
    .reset_index()
    .merge(
        pilot_df[
            pilot_df["variant"].eq(
                "augmented"
            )
        ][
            [
                "example_id",
                "changed_original_facts",
                "changed_augmented_facts",
                "changed_best_utterances",
                "persona_text",
                "dialogue_text",
            ]
        ],
        on="example_id",
        how="left",
    )
    .sort_values(
        [
            "contradiction_votes",
            "example_id",
        ]
    )
)


display(
    augmentation_inspection[
        [
            "example_id",
            "augmentation_types_clean",
            "contradiction_votes",
            *judge_labels,
            "changed_original_facts",
            "changed_augmented_facts",
            "changed_best_utterances",
        ]
    ]
)

,example_id,augmentation_types_clean,contradiction_votes,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct,changed_original_facts,changed_augmented_facts,changed_best_utterances
12,pair_0161_augmented,[family_composition_swap],0,0,0,0,0,[i have three sisters.],[i have 2 sisters.],[yes ! congratulations ! my three sisters are ...
27,pair_0214_augmented,[preference_negation],0,0,0,0,0,"[i like to read., i like cupcakes.]","[i do not like to read., i do not like cupcakes.]","[oh , do you enjoy reading ? me too ., does th..."
24,pair_0263_augmented,[favorite_swap],0,0,0,0,0,[my favorite color is blue.],[my favorite color is hunter green.],[what is your favorite color ? mine is blue .]
3,pair_0313_augmented,[profession_swap],0,0,0,0,0,[i work as a teacher and love my job.],[i work as a farmer.],"[hi , i am also a teacher ! love it . what age..."
46,pair_0367_augmented,[preference_negation],0,0,0,0,0,[i like to bake.],[i do not like to bake.],[thanks any way i love baking jelly cake what ...
22,pair_0591_augmented,"[family_composition_swap, count_swap]",0,0,0,0,0,"[i have 2 kids., i have 3 dog.]","[i have 3 kids., i have 6 dog.]","[wish i could my two kids love him, oh good lu..."
36,pair_0626_augmented,[preference_negation],0,0,0,0,0,[i enjoy playing retro video games on my 386.],[i do not enjoy playing retro video games on m...,"[i like old 386 video games more , but i used ..."
26,pair_0703_augmented,"[preference_negation, favorite_swap, professio...",0,0,0,0,0,"[i hate cleaning., my favorite color is blue.,...","[i love cleaning., my favorite color is black....",[i bet its easy to clean at least ! i need a c...
35,pair_0742_augmented,[location_swap],0,0,0,0,0,[i m from california.],[i m from the state of florida.],[that is nice . where do you live ? i live in ...
38,pair_0779_augmented,[favorite_swap],0,0,0,0,0,[my favorite food is steak.],[my favorite food is macaroni and cheese.],[ll . bubble gum . i love me some steak]


In [91]:
augmentation_inspection[
    augmentation_inspection[
        "contradiction_votes"
    ].le(1)
][
    [
        "example_id",
        "augmentation_types_clean",
        "contradiction_votes",
        "changed_original_facts",
        "changed_augmented_facts",
        "changed_best_utterances",
    ]
]

,example_id,augmentation_types_clean,contradiction_votes,changed_original_facts,changed_augmented_facts,changed_best_utterances
12,pair_0161_augmented,[family_composition_swap],0,[i have three sisters.],[i have 2 sisters.],[yes ! congratulations ! my three sisters are ...
27,pair_0214_augmented,[preference_negation],0,"[i like to read., i like cupcakes.]","[i do not like to read., i do not like cupcakes.]","[oh , do you enjoy reading ? me too ., does th..."
24,pair_0263_augmented,[favorite_swap],0,[my favorite color is blue.],[my favorite color is hunter green.],[what is your favorite color ? mine is blue .]
3,pair_0313_augmented,[profession_swap],0,[i work as a teacher and love my job.],[i work as a farmer.],"[hi , i am also a teacher ! love it . what age..."
46,pair_0367_augmented,[preference_negation],0,[i like to bake.],[i do not like to bake.],[thanks any way i love baking jelly cake what ...
22,pair_0591_augmented,"[family_composition_swap, count_swap]",0,"[i have 2 kids., i have 3 dog.]","[i have 3 kids., i have 6 dog.]","[wish i could my two kids love him, oh good lu..."
36,pair_0626_augmented,[preference_negation],0,[i enjoy playing retro video games on my 386.],[i do not enjoy playing retro video games on m...,"[i like old 386 video games more , but i used ..."
26,pair_0703_augmented,"[preference_negation, favorite_swap, professio...",0,"[i hate cleaning., my favorite color is blue.,...","[i love cleaning., my favorite color is black....",[i bet its easy to clean at least ! i need a c...
35,pair_0742_augmented,[location_swap],0,[i m from california.],[i m from the state of florida.],[that is nice . where do you live ? i live in ...
38,pair_0779_augmented,[favorite_swap],0,[my favorite food is steak.],[my favorite food is macaroni and cheese.],[ll . bubble gum . i love me some steak]


In [92]:
augmentation_inspection[
    augmentation_inspection[
        "contradiction_votes"
    ].eq(4)
][
    [
        "example_id",
        "augmentation_types_clean",
        "changed_original_facts",
        "changed_augmented_facts",
        "changed_best_utterances",
    ]
]

,example_id,augmentation_types_clean,changed_original_facts,changed_augmented_facts,changed_best_utterances
20,pair_0336_augmented,"[profession_swap, preference_negation]","[i work as a public affairs liaison., i love c...","[i work as a scientist., i hate cold , wintry ...",[what do you do ? i myself an a national guard...
16,pair_0480_augmented,"[preference_negation, favorite_swap]","[i enjoy the outdoors., my favorite breakfast ...","[i do not enjoy the outdoors., my favorite bre...","[not really , although i do like being outdoor..."
48,pair_0996_augmented,"[profession_swap, preference_negation, favorit...","[i work at mcdonalds., i like basketball., my ...","[i work at google., i do not like basketball.,...",[fabulous ! i am called gerald . i just finish...


In [93]:
PROMPT_AUDIT_IDS = [
    "pair_0161_augmented",
    "pair_0263_augmented",
    "pair_0367_augmented",
    "pair_0591_augmented",
    "pair_0998_augmented",
]


for example_id in PROMPT_AUDIT_IDS:
    row = pilot_df[
        pilot_df["example_id"].eq(
            example_id
        )
    ].iloc[0]

    print("=" * 100)
    print(example_id)
    print("=" * 100)

    print(
        build_judge_prompt(
            persona=row["persona_text"],
            dialogue=row["dialogue_text"],
        )
    )

    print()

pair_0161_augmented
You are a binary judge of persona consistency.

Determine whether statements made by SELF in the dialogue contradict at least one statement in SELF's persona.

Rules:
- CONTRADICTION: SELF explicitly states or clearly implies information that cannot be true at the same time as at least one persona statement.
- NO_CONTRADICTION: no such conflict is present.
- One conflicting persona statement is sufficient for CONTRADICTION.
- A persona statement that is not discussed in the dialogue is not a contradiction.
- Claims made only by PARTNER are not evidence about SELF. Use PARTNER's utterances only as conversational context.
- Respect temporal and modal information. Past and present statements, intentions, wishes, and current facts are contradictions only when they are genuinely incompatible.
- Additional compatible information is not a contradiction.
- Use only the information in the persona and dialogue. Do not assume unstated facts.

Persona of SELF:
- i live in a stu

In [94]:
CLEAR_LOW_VOTE_IDS = [
    "pair_0161_augmented",
    "pair_0214_augmented",
    "pair_0263_augmented",
    "pair_0367_augmented",
    "pair_0591_augmented",
    "pair_0626_augmented",
    "pair_0703_augmented",
    "pair_0998_augmented",
    "pair_0050_augmented",
    "pair_0061_augmented",
    "pair_0219_augmented",
    "pair_0368_augmented",
    "pair_0472_augmented",
    "pair_0577_augmented",
]


def normalize_evidence_list(
    value,
) -> list[str]:
    if value is None:
        return []

    if hasattr(
        value,
        "tolist",
    ):
        value = value.tolist()

    if isinstance(
        value,
        str,
    ):
        return [value]

    return [
        str(item)
        for item in value
    ]


evidence_only_df = (
    pilot_df[
        pilot_df["example_id"].isin(
            CLEAR_LOW_VOTE_IDS
        )
    ]
    .copy()
)


evidence_only_df[
    "dialogue_text"
] = (
    evidence_only_df[
        "changed_best_utterances"
    ]
    .map(
        lambda evidence:
            "\n".join(
                f"SELF: {utterance}"
                for utterance
                in normalize_evidence_list(
                    evidence
                )
            )
    )
)


evidence_only_df[
    "pair_id"
] = (
    evidence_only_df[
        "example_id"
    ]
)

evidence_only_df[
    "dialogue_id"
] = (
    evidence_only_df[
        "example_id"
    ]
)

evidence_only_df[
    "variant"
] = "evidence_only"


EVIDENCE_ONLY_PATH = (
    DATA_DIR
    / "llm_judge_evidence_only_diagnostic_v1.parquet"
)


evidence_only_df.to_parquet(
    EVIDENCE_ONLY_PATH,
    index=False,
)


print(
    "Examples:",
    len(
        evidence_only_df
    ),
)

display(
    evidence_only_df[
        [
            "example_id",
            "changed_augmented_facts",
            "dialogue_text",
        ]
    ]
)

Examples: 14


,example_id,changed_augmented_facts,dialogue_text
7,pair_0472_augmented,[i do not enjoy reading.],SELF: hi ! jen here . medical receptionist and...
18,pair_0998_augmented,[i hate watching sitcoms on television.],SELF: i like to watch funny sitcoms and read t...
25,pair_0161_augmented,[i have 2 sisters.],SELF: yes ! congratulations ! my three sisters...
30,pair_0368_augmented,[i m 43 years old.],"SELF: 53 , alabama . i am way older than you ...."
35,pair_0050_augmented,[i do not like to read.],SELF: well i am a reader . i love reading ende...
43,pair_0591_augmented,"[i have 3 kids., i have 6 dog.]",SELF: wish i could my two kids love him\nSELF:...
45,pair_0061_augmented,[i own 4 cats.],SELF: hey i got 3 cats and candy is favorite !...
50,pair_0263_augmented,[my favorite color is hunter green.],SELF: what is your favorite color ? mine is bl...
52,pair_0703_augmented,"[i love cleaning., my favorite color is black....",SELF: i bet its easy to clean at least ! i nee...
54,pair_0214_augmented,"[i do not like to read., i do not like cupcakes.]","SELF: oh , do you enjoy reading ? me too .\nSE..."


In [98]:
print(evidence_only_df.shape)

print(
    evidence_only_df[
        [
            "example_id",
            "changed_augmented_facts",
            "dialogue_text",
            "expected_label",
        ]
    ].to_string(index=False)
)


EVIDENCE_ONLY_PATH = (
    DATA_DIR
    / "llm_judge_evidence_only_diagnostic_v1.parquet"
)

evidence_only_df.to_parquet(
    EVIDENCE_ONLY_PATH,
    index=False,
)

print(
    f"Saved {len(evidence_only_df)} examples "
    f"to {EVIDENCE_ONLY_PATH}"
)

(14, 39)
         example_id                                                                 changed_augmented_facts                                                                                                                                                                                                                                                  dialogue_text expected_label
pair_0472_augmented                                                               [i do not enjoy reading.]                                                                                                                                                                                           SELF: hi ! jen here . medical receptionist and avid reader and you ?  CONTRADICTION
pair_0998_augmented                                                [i hate watching sitcoms on television.]                                                                                                                                

In [97]:
pilot_prediction_matrix = pd.DataFrame(
    {
        JUDGE_CONFIGS[
            model_id
        ]["label"]: (
            judgments
            .set_index(
                "example_id"
            )[
                "parsed_label_binary"
            ]
        )
        for model_id, judgments
        in pilot_judges.items()
    }
)


pilot_prediction_matrix = (
    pilot_prediction_matrix
    .sort_index()
)

pilot_prediction_matrix.head()

,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
example_id,,,,
pair_0040_augmented,1,0,1,0
pair_0040_original,1,0,1,0
pair_0044_augmented,1,0,1,0
pair_0044_original,0,0,1,0
pair_0050_augmented,0,0,1,0


In [76]:
from sklearn.metrics import cohen_kappa_score


def pairwise_agreement_matrices(
    predictions: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    Compute pairwise raw agreement and Cohen's kappa
    between all judge columns.

    Missing predictions are excluded pairwise.
    """
    judges = list(
        predictions.columns
    )

    raw_agreement = pd.DataFrame(
        index=judges,
        columns=judges,
        dtype=float,
    )

    kappa = pd.DataFrame(
        index=judges,
        columns=judges,
        dtype=float,
    )

    for judge_a in judges:
        for judge_b in judges:

            # A judge agrees perfectly with itself.
            if judge_a == judge_b:
                raw_agreement.loc[
                    judge_a,
                    judge_b,
                ] = 1.0

                kappa.loc[
                    judge_a,
                    judge_b,
                ] = 1.0

                continue

            common = (
                predictions[
                    [
                        judge_a,
                        judge_b,
                    ]
                ]
                .dropna()
            )

            if common.empty:
                continue

            values_a = common[
                judge_a
            ]

            values_b = common[
                judge_b
            ]

            raw_agreement.loc[
                judge_a,
                judge_b,
            ] = (
                values_a
                == values_b
            ).mean()

            kappa.loc[
                judge_a,
                judge_b,
            ] = cohen_kappa_score(
                values_a,
                values_b,
            )

    return (
        raw_agreement,
        kappa,
    )


(
    raw_agreement_matrix,
    kappa_matrix,
) = pairwise_agreement_matrices(
    pilot_prediction_matrix
)

print("Pairwise raw agreement:")
display(
    raw_agreement_matrix.style.format(
        "{:.1%}"
    )
)

print("Pairwise Cohen's kappa:")
display(
    kappa_matrix.style.format(
        "{:.3f}"
    )
)

Pairwise raw agreement:


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,100.0%,79.0%,61.0%,78.0%
Phi-4-mini-instruct,79.0%,100.0%,56.0%,89.0%
Mistral-7B-Instruct-v0.3,61.0%,56.0%,100.0%,55.0%
Llama-3.2-3B-Instruct,78.0%,89.0%,55.0%,100.0%


Pairwise Cohen's kappa:


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,1.000,0.307,0.242,0.263
Phi-4-mini-instruct,0.307,1.000,0.161,0.293
Mistral-7B-Instruct-v0.3,0.242,0.161,1.000,0.143
Llama-3.2-3B-Instruct,0.263,0.293,0.143,1.000


In [77]:
pilot_metadata = (
    pilot_df[
        [
            "example_id",
            "pair_id",
            "variant",
        ]
    ]
    .drop_duplicates(
        "example_id"
    )
    .set_index(
        "example_id"
    )
)


for variant in [
    "original",
    "augmented",
]:
    variant_ids = (
        pilot_metadata[
            pilot_metadata[
                "variant"
            ].eq(
                variant
            )
        ].index
    )

    variant_predictions = (
        pilot_prediction_matrix
        .loc[
            pilot_prediction_matrix
            .index
            .intersection(
                variant_ids
            )
        ]
    )

    (
        variant_raw,
        variant_kappa,
    ) = pairwise_agreement_matrices(
        variant_predictions
    )

    print(
        f"\n{variant.upper()} "
        "— raw agreement"
    )

    display(
        variant_raw.style.format(
            "{:.1%}"
        )
    )

    print(
        f"{variant.upper()} "
        "— Cohen's kappa"
    )

    display(
        variant_kappa.style.format(
            "{:.3f}"
        )
    )


ORIGINAL — raw agreement


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,100.0%,86.0%,60.0%,84.0%
Phi-4-mini-instruct,86.0%,100.0%,66.0%,98.0%
Mistral-7B-Instruct-v0.3,60.0%,66.0%,100.0%,68.0%
Llama-3.2-3B-Instruct,84.0%,98.0%,68.0%,100.0%


ORIGINAL — Cohen's kappa


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,1.000,0.000,-0.040,-0.036
Phi-4-mini-instruct,0.000,1.000,0.000,0.000
Mistral-7B-Instruct-v0.3,-0.040,0.000,1.000,0.076
Llama-3.2-3B-Instruct,-0.036,0.000,0.076,1.000



AUGMENTED — raw agreement


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,100.0%,72.0%,62.0%,72.0%
Phi-4-mini-instruct,72.0%,100.0%,46.0%,80.0%
Mistral-7B-Instruct-v0.3,62.0%,46.0%,100.0%,42.0%
Llama-3.2-3B-Instruct,72.0%,80.0%,42.0%,100.0%


AUGMENTED — Cohen's kappa


,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct
Qwen3-4B,1.000,0.338,0.313,0.323
Phi-4-mini-instruct,0.338,1.000,0.157,0.258
Mistral-7B-Instruct-v0.3,0.313,0.157,1.000,0.119
Llama-3.2-3B-Instruct,0.323,0.258,0.119,1.000


In [78]:
consensus_df = (
    pilot_metadata
    .join(
        pilot_prediction_matrix
    )
    .copy()
)


judge_columns = list(
    pilot_prediction_matrix.columns
)


consensus_df[
    "num_valid_judges"
] = (
    consensus_df[
        judge_columns
    ]
    .notna()
    .sum(
        axis=1
    )
)


consensus_df[
    "num_contradiction_votes"
] = (
    consensus_df[
        judge_columns
    ]
    .sum(
        axis=1,
        skipna=True,
    )
)


consensus_df[
    "num_no_contradiction_votes"
] = (
    consensus_df[
        "num_valid_judges"
    ]
    - consensus_df[
        "num_contradiction_votes"
    ]
)


complete_consensus_df = (
    consensus_df[
        consensus_df[
            "num_valid_judges"
        ].eq(
            len(
                judge_columns
            )
        )
    ]
    .copy()
)


complete_consensus_df[
    "vote_pattern"
] = (
    complete_consensus_df[
        "num_contradiction_votes"
    ]
    .astype(int)
    .astype(str)
    + " C / "
    + complete_consensus_df[
        "num_no_contradiction_votes"
    ]
    .astype(int)
    .astype(str)
    + " N"
)


complete_consensus_df[
    "unanimous"
] = (
    complete_consensus_df[
        "num_contradiction_votes"
    ].isin(
        [
            0,
            len(
                judge_columns
            ),
        ]
    )
)


print(
    "Examples with predictions "
    "from all judges:",
    len(
        complete_consensus_df
    ),
)

print(
    "Unanimous agreement:",
    f"{complete_consensus_df['unanimous'].mean():.1%}",
)

print(
    "\nVote patterns:"
)

display(
    complete_consensus_df[
        "vote_pattern"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "examples"
    )
)

Examples with predictions from all judges: 100
Unanimous agreement: 44.0%

Vote patterns:


,examples
vote_pattern,
0 C / 4 N,41
1 C / 3 N,35
2 C / 2 N,14
3 C / 1 N,7
4 C / 0 N,3


In [79]:
complete_consensus_df[
    "minority_size"
] = (
    complete_consensus_df[
        [
            "num_contradiction_votes",
            "num_no_contradiction_votes",
        ]
    ]
    .min(
        axis=1
    )
)


max_disagreement = (
    complete_consensus_df[
        ~complete_consensus_df[
            "unanimous"
        ]
    ]
    .sort_values(
        [
            "minority_size",
            "variant",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


disagreement_inspection = (
    max_disagreement
    .reset_index()
    .merge(
        pilot_df[
            [
                "example_id",
                "expected_label",
                "changed_original_facts",
                "changed_augmented_facts",
            ]
        ]
        .drop_duplicates(
            "example_id"
        ),
        on="example_id",
        how="left",
    )
)


display_columns = (
    [
        "example_id",
        "variant",
        "expected_label",
        "vote_pattern",
    ]
    + judge_columns
    + [
        "changed_original_facts",
        "changed_augmented_facts",
    ]
)


disagreement_inspection[
    display_columns
].head(20)

,example_id,variant,expected_label,vote_pattern,Qwen3-4B,Phi-4-mini-instruct,Mistral-7B-Instruct-v0.3,Llama-3.2-3B-Instruct,changed_original_facts,changed_augmented_facts
0,pair_0473_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,"[i like to go running., i love to dance.]","[i do not like to go running., i hate to dance.]"
1,pair_0502_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,"[i live with my boyfriend of 5 years., i was b...","[i live alone., i was born in pittsburgh.]"
2,pair_0370_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,[i love chocolate.],[i hate chocolate.]
3,pair_0649_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,[i work in a diner.],"[i work in the office , 9 till 5.]"
4,pair_0040_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,"[i love to sing., i have 3 children.]","[i hate to sing., i have 5 children.]"
5,pair_0599_augmented,augmented,CONTRADICTION,2 C / 2 N,0,1,1,0,[i am ten years old.],[i am 20 years old.]
6,pair_0783_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,[i like to listen to rap music.],[i do not like to listen to rap music.]
7,pair_0447_augmented,augmented,CONTRADICTION,2 C / 2 N,0,0,1,1,[i like to draw.],[i do not like to draw.]
8,pair_0952_augmented,augmented,CONTRADICTION,2 C / 2 N,1,0,1,0,[i have played since i was 4 years old.],[i have played since i was 14 years old.]
9,pair_0862_augmented,augmented,CONTRADICTION,2 C / 2 N,0,1,1,0,"[i am vegan., i work at the grocery store.]","[i eat meat., i work at a museum.]"
